In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
from osgeo import gdal
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import joblib # guardar y cargar el modelo

# --- 1. PATH CONFIGURATION ---
# Reemplaza estas variables por las rutas correctas en tu entorno (Drive/Colab)
base_folder = '/content/drive/MyDrive/ann_colab/'
BASE_MAP_PATH = base_folder + 'mapa_base_2014.tif'
SAMPLES_CSV_PATH = base_folder + 'muestras_2014_s.csv' # Tu CSV con 'Longitude', 'Latitude', 'LandCover'
OUTPUT_CSV_PATH = base_folder + 'clasificacion_2014_resultados.csv'
OUTPUT_TIF_PATH = base_folder + 'mapa_clasificado_2014.tif'

# --- 2. RASTER LOADING AND PREPROCESSING ---
print("--- Loading base map and processing bands ---")
dataset = gdal.Open(BASE_MAP_PATH)
geotransform = dataset.GetGeoTransform()
projection = dataset.GetProjection()

n_cols = dataset.RasterXSize
n_rows = dataset.RasterYSize
n_bands = dataset.RasterCount

# Leer todas las 18 bandas como un array 3D (bandas, filas, columnas)
raster_data = dataset.ReadAsArray()

# Crear una máscara para identify valid pixels.
# Inicialmente todos son True, pero we will discard any containing NaN
valid_mask = np.ones((n_rows, n_cols), dtype=bool)

for i in range(n_bands):
    band_array = raster_data[i]
    nodata_val = dataset.GetRasterBand(i + 1).GetNoDataValue()

    # Discard pixels with native numpy NaN
    valid_mask &= ~np.isnan(band_array)
    # Discard specific NoData values configured in the TIF (si existen)
    if nodata_val is not None:
        valid_mask &= (band_array != nodata_val)

print(f"Total pixels in the bounding box: {n_rows * n_cols}")
print(f"Valid pixels (without Nulls/NaN) for the irregular map: {np.sum(valid_mask)}")

# --- 3. EXTRACTION OF TRAINING VALUES FROM CSV ---
print("\n--- Loading samples from CSV and intersecting with TIF ---")
df_samples = pd.read_csv(SAMPLES_CSV_PATH)

X_train_list = []
y_train_list = []

for index, row in df_samples.iterrows():
    lon = row['Longitude']
    lat = row['Latitude']
    land_cover = int(row['LandCover'])

    # Convert UTM coordinates (EPSG:32719) to Row and Column indices
    col = int((lon - geotransform[0]) / geotransform[1])
    row = int((lat - geotransform[3]) / geotransform[5])

    # Check that the point is within the image and falls in a valid cell (not NaN)
    if 0 <= col < n_cols and 0 <= row < n_rows:
        if valid_mask[row, col]:
            # Extract the 18 variables
            band_values = raster_data[:, row, col]
            X_train_list.append(band_values)
            y_train_list.append(land_cover)

X = np.array(X_train_list)
y = np.array(y_train_list)

print(f"Extracted {len(y)} valid spectral signatures from {len(df_samples)} points in the CSV.")

# --- 4. TRAINING AND HYPERPARAMETER SEARCH ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalar (Normalizar) los datos es crítico.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Configuring Hyperparameter Search ---")

parameter_space = {
    'hidden_layer_sizes': [
        (100,), (128,), (150,),                         # 1 capa
        (100, 50), (128, 64), (150, 75), (100, 100),   # 2 capas
        (128, 64, 32), (100, 50, 25), (150, 75, 37),   # 3 capas
        (256, 128, 64), (128, 128, 64),                # 4+ capas
    ],
    'learning_rate_init': [0.005, 0.008, 0.01, 0.012, 0.015, 0.02],
    'alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'solver': ['adam'],  # Solo Adam
    'batch_size': [16, 32, 64],
}

base_mlp = MLPClassifier(
    max_iter=2000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    tol=1e-5,
    warm_start=False,  # Resetear pesos cada vez
)

search = RandomizedSearchCV(
    estimator=base_mlp,
    param_distributions=parameter_space,
    n_iter=250,  # More iterations
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("\n Entrenando y buscando el mejor modelo (esto tomará un momento)...")
search.fit(X_train_scaled, y_train)

# 4. Obtener el modelo óptimo
best_mlp = search.best_estimator_

# Class analysis problemática
from sklearn.metrics import classification_report, confusion_matrix
y_pred = best_mlp.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("\n==================================================")
print(" SEARCH COMPLETED! Best Hyperparameters:")
print(search.best_params_)
print("==================================================")

# --- NEW: SHOW TOP 20 BEST MODELS ---
print("\n TOP 20 BEST TESTED CONFIGURATIONS ")
print("-" * 85)
# Convert results to DataFrame for easy manipulation
cv_results = pd.DataFrame(search.cv_results_)

# Sort results by rank and take top 20
top_20 = cv_results.sort_values(by='rank_test_score').head(20)

for index, row in top_20.iterrows():
    # Show rank, mean score (accuracy), standard deviation, and parameters
    print(f"Rank {row['rank_test_score']:02d} | Accuracy: {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})")
    print(f"Parameters: {row['params']}")
    print("-" * 85)

# Validation and report with the BEST model
y_pred_test = best_mlp.predict(X_test_scaled)
print(f"\nInternal Global Accuracy of the best model: {accuracy_score(y_test, y_pred_test):.2%}")
print("\nClassification Report (Test Split):")
print(classification_report(y_test, y_pred_test))

# --- NEW: SAVE MODEL AND SCALER ---
print("\n--- Saving optimal model to disk ---")
MODEL_PATH = base_folder + 'mejor_modelo_mlp_2014.pkl'
SCALER_PATH = base_folder + 'scaler_mlp_2014.pkl' # Saving the scaler is MANDATORY

joblib.dump(best_mlp, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

print(f" Model saved at: {MODEL_PATH}")
print(f" Scaler saved at: {SCALER_PATH}")

# ... (El resto de tu código paso 5, 6 y 7 sigue igual, pero usando 'best_mlp' en lugar de 'mlp') ...

# --- 5. FULL MAP CLASSIFICATION ---
print("\n--- Classifying the full map (only valid zones) ---")
# Extract only True cells from valid_mask. Transpose for table format
valid_pixels = raster_data[:, valid_mask].T

# Scale using the same training scaler
valid_pixels_scaled = scaler.transform(valid_pixels)

# Final classification
predictions = best_mlp.predict(valid_pixels_scaled)

# --- 6. EXPORT RESULTS TO CSV ---
print(f"\n--- Exportando predictions a CSV: {OUTPUT_CSV_PATH} ---")
# Get matrix coordinates (Y, X) of valid pixels
row_idx, col_idx = np.where(valid_mask)

# Reconstruir las coordenadas UTM a partir del índice de columna/row
# Add half pixel size to reference cell center
valid_lon = geotransform[0] + col_idx * geotransform[1] + (geotransform[1] / 2)
valid_lat = geotransform[3] + row_idx * geotransform[5] + (geotransform[5] / 2)

df_results = pd.DataFrame({
    'Longitude': valid_lon,
    'Latitude': valid_lat,
    'LandCover': predictions
})
df_results.to_csv(OUTPUT_CSV_PATH, index=False)
print(f" Archivo CSV guardado. Generó {len(predictions)} registros.")

# --- 7. EXPORT TIF RASTER (FOR QGIS) ---
print(f"\n--- Generating classified map for QGIS: {OUTPUT_TIF_PATH} ---")
# Initialize matrix filled with ZEROS (empty space outside irregular map)
classified_map = np.zeros((n_rows, n_cols), dtype=np.uint8)

# Fill with classes (1, 2, 3) only valid zones
classified_map[valid_mask] = predictions

# Export as GeoTIFF
driver = gdal.GetDriverByName('GTiff')
out_dataset = driver.Create(OUTPUT_TIF_PATH, n_cols, n_rows, 1, gdal.GDT_Byte)
out_dataset.SetGeoTransform(geotransform)
out_dataset.SetProjection(projection)

out_band = out_dataset.GetRasterBand(1)
out_band.WriteArray(classified_map)
# Strictly indicate to QGIS/GDAL that value 0 is transparent/NoData
out_band.SetNoDataValue(0)
out_dataset.FlushCache()
out_dataset = None

print(" TIF Map saved. Finished!")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import pandas as pd
import numpy as np

# --- 4. DETAILED EVALUATION ---

# Assume we already have: y_test (actual) and y_pred_test (predicted)
labels = [1, 2, 3]
target_names = ["1: Urban", "2: Vegetation", "3: Soil"]

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test, labels=labels)

# 2. Metrics by Class (User's Accuracy, Producer's Accuracy, F1)
report_dict = classification_report(y_test, y_pred_test, target_names=target_names, output_dict=True)

# 3. Manual Kappa calculation
N = np.sum(cm)
Po = accuracy_score(y_test, y_pred_test) # Observed agreement (Global Accuracy)

# Totales por row (reales) y columna (predichos)
row_sums = np.sum(cm, axis=1) # Actual Marginals
col_sums = np.sum(cm, axis=0) # Predicted Marginals

# Pe (Agreement by chance)
Pe = np.sum((row_sums * col_sums)) / (N**2)

# P_quantity (Maximum possible agreement given histogram)
# Based on the minimum of marginals for each class
P_quantity = np.sum(np.minimum(row_sums, col_sums)) / N

# Kappa Indices
k_standard = (Po - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0
k_histo = (P_quantity - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0
k_loc = (Po - P_quantity) / (1 - P_quantity) if (1 - P_quantity) != 0 else 0

# --- RESULTS PRESENTATION ---

print(f"\n{'='*60}")
print(" VALIDATION RESULTS (TEST SET) ")
print(f"{'='*60}")

print(f"\n GLOBAL METRICS:")
print(f"   - Global Accuracy (OA):   {Po:.2%}")
print(f"   - Standard Kappa:          {k_standard:.4f}")
print(f"   - Histogram Kappa (Kh):   {k_histo:.4f}  <-- (Quantity agreement)")
print(f"   - Location Kappa (Kl): {k_loc:.4f}  <-- (Location agreement)")

print(f"\n METRICS BY LAND COVER TYPE:")
# Build a table for readability
class_table = []
for i, name in enumerate(target_names):
    class_id = labels[i]
    # User Accuracy (Precision): How reliable is the map?
    ua = report_dict[name]['precision']
    # Producer Accuracy (Recall): How well was reality identified?
    pa = report_dict[name]['recall']
    f1 = report_dict[name]['f1-score']

    class_table.append({
        "LandCover": name,
        "User Acc. (Prec)": f"{ua:.2%}",
        "Prod Acc. (Rec)": f"{pa:.2%}",
        "F1-Score": f"{f1:.4f}"
    })

df_metrics = pd.DataFrame(class_table)
print(df_metrics.to_string(index=False))

print(f"\n CONFUSION MATRIX (Row=Actual, Col=Predicted):")
df_cm = pd.DataFrame(cm, index=[f"R_{n}" for n in target_names],
                         columns=[f"P_{n}" for n in target_names])
print(df_cm)

In [ ]:
import numpy as np
from osgeo import gdal
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import joblib
import random

# --- 1. PATH CONFIGURATION ---
base_folder = '/content/drive/MyDrive/ann_colab/'
BASE_MAP_PATH = base_folder + 'mapa_base_2014.tif'
SAMPLES_CSV_PATH = base_folder + 'muestras_2014_s.csv'
OUTPUT_CSV_PATH = base_folder + 'clasificacion_2014_definitivo.csv'
OUTPUT_TIF_PATH = base_folder + 'mapa_clasificado_2014_definitivo.tif'

# Model paths (Read and Write)
OPTIMAL_MODEL_PATH = base_folder + 'mejor_modelo_mlp_2014.pkl'
FINAL_MODEL_PATH = base_folder + 'modelo_definitivo_mlp_2014.pkl'
FINAL_SCALER_PATH = base_folder + 'scaler_definitivo_mlp_2014.pkl'

# --- 2. RASTER LOADING AND PREPROCESSING ---
print("--- Loading base map and processing bands ---")
dataset = gdal.Open(BASE_MAP_PATH)
geotransform = dataset.GetGeoTransform()
projection = dataset.GetProjection()

n_cols = dataset.RasterXSize
n_rows = dataset.RasterYSize
n_bands = dataset.RasterCount

raster_data = dataset.ReadAsArray()
valid_mask = np.ones((n_rows, n_cols), dtype=bool)

for i in range(n_bands):
    band_array = raster_data[i]
    nodata_val = dataset.GetRasterBand(i + 1).GetNoDataValue()
    valid_mask &= ~np.isnan(band_array)
    if nodata_val is not None:
        valid_mask &= (band_array != nodata_val)

print(f"Valid pixels for the map: {np.sum(valid_mask)}")

# --- 3. TRAINING VALUES EXTRACTION ---
print("\n--- Loading samples from CSV ---")
df_samples = pd.read_csv(SAMPLES_CSV_PATH)

X_train_list, y_train_list = [], []

for index, row in df_samples.iterrows():
    lon, lat, land_cover = row['Longitude'], row['Latitude'], int(row['LandCover'])
    col = int((lon - geotransform[0]) / geotransform[1])
    row = int((lat - geotransform[3]) / geotransform[5])

    if 0 <= col < n_cols and 0 <= row < n_rows and valid_mask[row, col]:
        X_train_list.append(raster_data[:, row, col])
        y_train_list.append(land_cover)

X = np.array(X_train_list)
y = np.array(y_train_list)

# --- 4. NESTED EXPERIMENT: SAMPLES VS WEIGHTS ---
print("\n--- Loading base model optimal parameters ---")
# Load the optimal model obtained in RandomizedSearchCV
previous_model = joblib.load(OPTIMAL_MODEL_PATH)
optimal_parameters = previous_model.get_params()

# Experiment configuration
num_sample_groups = 5  # How many different data partitions to test
num_weight_inits = 5    # How many weight initializations per data group

# Variables to save the absolute champion of ALL tests
best_global_accuracy = 0.0
champion_mlp = None
champion_scaler = None
champion_split_seed = None
champion_weights_seed = None

print(f"\n Starting Experiment: {num_sample_groups} Data Groups x {num_weight_inits} Weight Inits ({num_sample_groups * num_weight_inits} total trainings)")
print("-" * 80)

for i in range(num_sample_groups):
    # 1. Generate a random seed for data partition (The Group)
    split_seed = random.randint(1, 10000)

    # Split data for this group
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=split_seed, stratify=y
    )

    # Scale data specifically for this group
    group_scaler = StandardScaler()
    X_train_scaled = group_scaler.fit_transform(X_train)
    X_test_scaled = group_scaler.transform(X_test)

    # List to save the accuracies of the 5 weight tests in this group
    group_accuracies = []

    print(f" GROUP {i+1}/{num_sample_groups} | Split Seed: {split_seed}")

    # Inner loop: Test 5 different weight inits for THIS group
    for j in range(num_weight_inits):
        weights_seed = random.randint(1, 10000)

        # Inject weight seed into hyperparameters dict
        optimal_parameters['random_state'] = weights_seed
        test_mlp = MLPClassifier(**optimal_parameters)

        # Train and predict
        test_mlp.fit(X_train_scaled, y_train)
        y_pred = test_mlp.predict(X_test_scaled)
        current_acc = accuracy_score(y_test, y_pred)

        group_accuracies.append(current_acc)

        # Evaluate if this specific model is the best of ALL seen
        if current_acc > best_global_accuracy:
            best_global_accuracy = current_acc
            champion_mlp = test_mlp
            champion_scaler = group_scaler
            champion_split_seed = split_seed
            champion_weights_seed = weights_seed

    # Calculate and print group average
    group_average = np.mean(group_accuracies)
    group_deviation = np.std(group_accuracies)
    print(f"    Promedio del Grupo {i+1}: {group_average:.4%} (+/- {group_deviation:.4%})")
    print("-" * 80)

print("\n==================================================")
print(" EXPERIMENT COMPLETED! ABSOLUTE CHAMPION FOUND ")
print(f"Best Individual Accuracy: {best_global_accuracy:.4%}")
print(f"Samples Seed (Split): {champion_split_seed}")
print(f"Weights Init Seed: {champion_weights_seed}")
print("==================================================")

# Save the champion model and its scaler
joblib.dump(champion_mlp, FINAL_MODEL_PATH)
joblib.dump(champion_scaler, FINAL_SCALER_PATH)

# --- NUEVO: DETAILED EVALUATION DEL MODELO CAMPEÓN ---
print("\n--- REBUILDING CHAMPION ENVIRONMENT FOR DETAILED METRICS ---")

# 1. Recreate exact split used by the champion model
_, X_test_champ, _, y_test_champ = train_test_split(
    X, y, test_size=0.2, random_state=champion_split_seed, stratify=y
)

# 2. Scale test data with champion scaler and predict
X_test_champ_scaled = champion_scaler.transform(X_test_champ)
y_pred_champ = champion_mlp.predict(X_test_champ_scaled)

# 3. Calcular las métricas usando y_test_champ (reales) y y_pred_champ (predichos)
labels = [1, 2, 3]
target_names = ["1: Urban", "2: Vegetation", "3: Soil"]

# Confusion Matrix
cm = confusion_matrix(y_test_champ, y_pred_champ, labels=labels)

# Metrics by Class
report_dict = classification_report(y_test_champ, y_pred_champ, target_names=target_names, output_dict=True)

# Manual Kappa calculation
N = np.sum(cm)
Po = accuracy_score(y_test_champ, y_pred_champ)

row_sums = np.sum(cm, axis=1)
col_sums = np.sum(cm, axis=0)

Pe = np.sum((row_sums * col_sums)) / (N**2)
P_quantity = np.sum(np.minimum(row_sums, col_sums)) / N

k_standard = (Po - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0
k_histo = (P_quantity - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0
k_loc = (Po - P_quantity) / (1 - P_quantity) if (1 - P_quantity) != 0 else 0

# --- RESULTS PRESENTATION ---
print(f"\n{'='*60}")
print(" DEFINITIVE MODEL VALIDATION RESULTS (TEST SET) ")
print(f"{'='*60}")

print(f"\n GLOBAL METRICS:")
print(f"   - Global Accuracy (OA):   {Po:.2%}")
print(f"   - Standard Kappa:          {k_standard:.4f}")
print(f"   - Histogram Kappa (Kh):   {k_histo:.4f}  <-- (Quantity agreement)")
print(f"   - Location Kappa (Kl): {k_loc:.4f}  <-- (Location agreement)")

print(f"\n METRICS BY LAND COVER TYPE:")
class_table = []
for i, name in enumerate(target_names):
    ua = report_dict[name]['precision']
    pa = report_dict[name]['recall']
    f1 = report_dict[name]['f1-score']

    class_table.append({
        "LandCover": name,
        "User Acc. (Prec)": f"{ua:.2%}",
        "Prod Acc. (Rec)": f"{pa:.2%}",
        "F1-Score": f"{f1:.4f}"
    })

df_metrics = pd.DataFrame(class_table)
print(df_metrics.to_string(index=False))

print(f"\n CONFUSION MATRIX (Row=Actual, Col=Predicted):")
df_cm = pd.DataFrame(cm, index=[f"R_{n}" for n in target_names],
                         columns=[f"P_{n}" for n in target_names])
print(df_cm)

# --- 5. FULL MAP CLASSIFICATION ---
print("\n--- Classifying full map with definitive model ---")
valid_pixels = raster_data[:, valid_mask].T
valid_pixels_scaled = champion_scaler.transform(valid_pixels) # Use the champion scaler
predictions = champion_mlp.predict(valid_pixels_scaled)

# --- 6. EXPORT RESULTS TO CSV ---
print(f"\n--- Exportando predictions a CSV: {OUTPUT_CSV_PATH} ---")
row_idx, col_idx = np.where(valid_mask)
valid_lon = geotransform[0] + col_idx * geotransform[1] + (geotransform[1] / 2)
valid_lat = geotransform[3] + row_idx * geotransform[5] + (geotransform[5] / 2)

df_results = pd.DataFrame({
    'Longitude': valid_lon,
    'Latitude': valid_lat,
    'LandCover': predictions
})
df_results.to_csv(OUTPUT_CSV_PATH, index=False)
print(f" Archivo CSV guardado. Generó {len(predictions)} registros.")

# --- 7. EXPORT TIF RASTER (FOR QGIS) ---
print(f"\n--- Generating classified map for QGIS: {OUTPUT_TIF_PATH} ---")
classified_map = np.zeros((n_rows, n_cols), dtype=np.uint8)
classified_map[valid_mask] = predictions

driver = gdal.GetDriverByName('GTiff')
out_dataset = driver.Create(OUTPUT_TIF_PATH, n_cols, n_rows, 1, gdal.GDT_Byte)
out_dataset.SetGeoTransform(geotransform)
out_dataset.SetProjection(projection)

out_band = out_dataset.GetRasterBand(1)
out_band.WriteArray(classified_map)
out_band.SetNoDataValue(0)
out_dataset.FlushCache()
out_dataset = None

print(" TIF map correctly generated. Process completed with the best seed!")